# DEX ARM 01: Train ACT Policy via LeRobot
**Dataset:** `salik/dex-arm-01-sim` (37 episodes, 1,722 frames, 30 FPS)
**Target Policy:** Action Chunking with Transformers (ACT)
**Checkpoint Hub Destination:** `salik/dex-arm-01-act`
---
> **Colab Runtime Requirement:** Connect to **T4 GPU** using the Antigravity Colab Extension.

In [ ]:
# 1. Verify GPU and Python Runtime
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys; print("python", sys.version.split()[0])
assert sys.version_info >= (3, 12), "lerobot 0.6.1 requires Python 3.12+"

In [ ]:
# 2. Install Pinned Dependencies
!pip install -q "lerobot[dataset,training]==0.6.1"
!pip install -q "av>=14,<17"
!lerobot-train --help | head -5

In [ ]:
# 3. Authenticate with Hugging Face (Write Token Required)
from huggingface_hub import login
import getpass
login(token=getpass.getpass("Enter your Hugging Face Write Token: "))

In [ ]:
# 4. Validate Dataset Contract Before Spending GPU Time
from huggingface_hub import HfApi
from lerobot.datasets.lerobot_dataset import LeRobotDataset

try:
    HfApi().create_tag("salik/dex-arm-01-sim", tag="v3.0", repo_type="dataset")
    print("Created codebase version tag v3.0 on Hugging Face.")
except Exception as e:
    print("Tag status:", type(e).__name__)

ds = LeRobotDataset("salik/dex-arm-01-sim")
print("episodes:", ds.num_episodes, "frames:", ds.num_frames, "fps:", ds.fps)
s = ds[0]
print("state shape:", tuple(s["observation.state"].shape))
print("env shape:", tuple(s["observation.environment_state"].shape))
print("action shape:", tuple(s["action"].shape))
assert ds.num_episodes == 37 and ds.num_frames == 1722, "Dataset mismatch! Stop here."

In [ ]:
# 5. Train ACT Policy (~30-90 min on T4 GPU)
!lerobot-train \
  --policy.type=act \
  --dataset.repo_id=salik/dex-arm-01-sim \
  --output_dir=outputs/act_dex_arm \
  --job_name=dex_arm_act \
  --steps=20000 \
  --batch_size=16 \
  --num_workers=2 \
  --save_freq=5000 \
  --wandb.enable=false \
  --policy.push_to_hub=true \
  --policy.repo_id=salik/dex-arm-01-act 2>&1 | tee /content/train.log

In [ ]:
# 6. Verify Checkpoint Landing on Hugging Face Hub
from huggingface_hub import HfApi
files = HfApi().list_repo_files("salik/dex-arm-01-act")
print(f"Total files in salik/dex-arm-01-act: {len(files)}")
for f in files:
    print(" -", f)
assert any(f.endswith("config.json") for f in files), "Checkpoint upload missing config.json!"
print("\nSUCCESS: DEX ARM 01 ACT Model Trained and Published to Hub!")